# HR Innovate GA4 V4 — acquisition sources and fair forecast comparison

**Default mode: SYNTHETIC FIXTURE, not business analytics.** No live extraction or accuracy improvement is claimed. Change the private input paths only when authorized GA4 data is available. Keep executed live notebooks under ignored `local-v4/`.

V3 remains archived. The adapter reproduces its pure helpers with a documented in-memory syntax repair. Outer evaluation uses identical seven-day horizons; model selection, bias adjustment and grouping use earlier inner folds. October 18–24 is a retrospective simulation with October 17 cutoff; no future data may influence selection.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'projects/01_ga4_traffic_forecasting').exists())
project = root / 'projects/01_ga4_traffic_forecasting'
sys.path.insert(0, str(project / 'src'))
import forecast_v4_ga4 as v4
CUTOFF = '2026-10-17'
SOURCES_CSV = None  # e.g. root / 'local-v4/sources.csv'
AGGREGATE_CSV = None  # independently exported daily totals
FIXTURE = SOURCES_CSV is None and AGGREGATE_CSV is None
if (SOURCES_CSV is None) != (AGGREGATE_CSV is None):
    raise ValueError('Both private input paths are required')
if FIXTURE:
    raw, aggregate = v4.synthetic_fixture()
else:
    raw, aggregate = pd.read_csv(SOURCES_CSV), pd.read_csv(AGGREGATE_CSV)
panel, total = v4.prepare(raw, aggregate, CUTOFF)
LABEL = 'SYNTHETIC FIXTURE — NOT HR INNOVATE RESULTS' if FIXTURE else 'Private GA4 data'
print(LABEL)
print(f'{len(total)} reconciled days, through {total.index.max().date()}')

## Source distribution and time patterns
Source, medium and channel group are retained separately. Classification is exclusive, with LinkedIn taking priority (including paid LinkedIn). These descriptive views do not prove causality or explain a forecast miss by themselves. EDA is restricted to the cutoff; it does not change the fixed model search space.

In [ ]:
past = raw.loc[pd.to_datetime(raw.date) <= pd.Timestamp(CUTOFF)]
source_mix = past.groupby(v4.DIMENSIONS[1:], dropna=False).sessions.sum().sort_values(ascending=False)
display(source_mix.to_frame('sessions'))
summary = pd.DataFrame({'sessions': panel.sum(), 'share_pct': panel.sum() / total.sum() * 100,
                        'daily_mean': panel.mean(), 'daily_std': panel.std(),
                        'zero_days': (panel == 0).sum()})
display(summary.sort_values('sessions', ascending=False))
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
panel.rolling(7).mean().plot(ax=axes[0], title=f'{LABEL} | 7-day mean channel traffic')
panel.groupby(panel.index.dayofweek).mean().plot.bar(ax=axes[1], title='Weekday patterns (Monday=0)')
plt.tight_layout()
plt.show()

In [ ]:
# Descriptive changes, not claims of statistically confirmed structural breaks.
display(pd.DataFrame({'prior_28_mean': panel.iloc[-56:-28].mean(),
                      'recent_28_mean': panel.iloc[-28:].mean()}))
# Contributions to unusual totals relative to the same weekday one week earlier.
change = panel - panel.shift(7)
unusual_dates = change.sum(axis=1, min_count=1).abs().nlargest(5).index
display(change.loc[unusual_dates].assign(total_change=change.loc[unusual_dates].sum(axis=1)))
if {'Direct', 'LinkedIn'}.issubset(panel.columns):
    panel[['Direct', 'LinkedIn']].tail(56).plot(figsize=(12, 4), title=f'{LABEL} | Direct and LinkedIn')
    plt.show()

## Nested model selection and evaluation
Default live protocol: eight outer weeks and eight inner weeks. The fixture uses four and two to keep demonstration runtime manageable. Compare matched outer holdouts only. MAPE omits zero-actual days; MAE includes them. Positive bias means overforecast. Weekly percentage error uses actual totals as denominator. Source predictability must be interpreted alongside scale and zero frequency, not MAPE alone.

In [ ]:
rows, source_rows, selections = v4.backtest(panel, total,
    outer_weeks=4 if FIXTURE else 8, inner_weeks=2 if FIXTURE else 8)
display(v4.summarize(rows, ['model']))
display(v4.summarize(rows, ['model', 'cutoff']))
display(v4.summarize(rows.assign(weekday=rows.date.dt.day_name()), ['model', 'weekday']))
display(v4.summarize(source_rows, ['channel']))
display(pd.DataFrame(selections))

In [ ]:
week = rows.loc[(rows.model == 'V4') & (rows.cutoff == rows.cutoff.max())].sort_values('date')
x = np.arange(len(week))
fig, ax = plt.subplots(figsize=(12, 6), facecolor='black')
ax.set_facecolor('black')
for shift, column, color, label in [(-.18, 'predicted', '#238bff', 'Predicted'), (.18, 'actual', '#24b76a', 'Actual')]:
    bars = ax.bar(x + shift, week[column], .36, color=color, label=label)
    ax.bar_label(bars, fmt='%.1f', color='white', padding=4)
ax.set_xticks(x, week.date.dt.strftime('%a %m/%d'), color='white')
ax.tick_params(axis='y', colors='white')
ax.grid(axis='y', linestyle='--', alpha=.35)
ax.set_axisbelow(True)
ax.set_ylim(0, week[['actual', 'predicted']].to_numpy().max() * 1.2)
for spine in ax.spines.values(): spine.set_visible(False)
fig.suptitle('V4 predicted versus actual sessions', color='white', fontsize=16)
ax.set_title(f'{LABEL}\nHistorical holdout: {week.date.min():%Y-%m-%d} to {week.date.max():%Y-%m-%d}', color='white', fontsize=10)
ax.legend(facecolor='black', labelcolor='white')
plt.tight_layout()
plt.show()

## October 18–24 retrospective simulation
The frozen historical V3 forecast is preserved in its own file and column. An adapted V3 re-fit is distinct. Fixture predictions below demonstrate the interface only and must never be copied into the business October forecast. Actual October traffic is not used here, including for model selection.

In [ ]:
channels, selection = v4.forecast_v4(panel, inner_weeks=2 if FIXTURE else 8)
adapted, selection_v3 = v4.forecast_v3(total, inner_weeks=2 if FIXTURE else 8)
simulation = pd.DataFrame({'date': channels.index, 'v4': channels.sum(axis=1).to_numpy(), 'v3_adapted': adapted})
original = pd.read_csv(project / 'reports/v3_forecast_oct18_24_2026.csv', parse_dates=['date'])
assert original.expected.tolist() == [57, 145, 137, 149, 125, 109, 58]
if not FIXTURE:
    simulation = simulation.merge(original[['date', 'expected']].rename(columns={'expected': 'v3_original'}), on='date', how='left')
print(LABEL, '| retrospective simulation, cutoff', CUTOFF)
print('V4 selection:', selection, '| V3 adapted:', selection_v3)
display(simulation)
display(channels)

## Interpretation and uncertainty
Synthetic results verify mechanics only; they cannot establish the most predictable real channel or a reduction toward 15%/10% MAPE. Live comparison, source rankings and October predictions remain blocked pending existing GA4 configuration. No prediction intervals or statistical significance are asserted from these fixtures. For live assessment, compare paired weekly losses on enough independent target weeks and retain a separate prospective period before adoption.

For a genuine prospective run, choose a completed cutoff in the GA4 property timezone and freeze forecasts before the first target date begins. If reporting latency prevents that, label the output as-of/retrospective. Use a new local output directory each time; score only after all seven days finalize. GA4 revisions mean a later extract is not an original as-of snapshot. See the comparison report for the protocol and limitations.